Import kaggle datasets

In [ ]:
from google.colab import files
files.upload()

Step 2: From kaggle dataset unzip IMBD Dataset

In [ ]:
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip imdb-dataset-of-50k-movie-reviews.zip

mkdir: cannot create directory ‘/root/.kaggle’: File exists
Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
  0% 0.00/25.7M [00:00<?, ?B/s]
100% 25.7M/25.7M [00:00<00:00, 1.03GB/s]
Archive:  imdb-dataset-of-50k-movie-reviews.zip
  inflating: IMDB Dataset.csv        


IMport required libraries for - data preprocessing, basic ml models and deep learning models

In [ ]:
import numpy as np
import pandas as pd

import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score,classification_report

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,Flatten,Dense,Dropout,LSTM,Conv1D,GlobalMaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
Read the csv file and display top 5 rows and number of positive and negative values

In [ ]:
df=pd.read_csv("IMDB Dataset.csv")
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


Changing all letters into lowercase, removing html tags, punction and numbers and bring all the words back to the root word

In [ ]:
stop_words=set(stopwords.words('english'))
lemmantizer=WordNetLemmatizer()
def clean_text(text):
  text=text.lower()
  text=re.sub(r'<.*?>',' ',text)
  text=re.sub(r'[^a-zA-Z]',' ',text)
  words=[lemmantizer.lemmatize(w) for w in text.split() if w not in stop_words]
  return ' '.join(words)
df['clean_review']=df['review'].apply(clean_text)
print(df['clean_review'].head())

0    one reviewer mentioned watching oz episode hoo...
1    wonderful little production filming technique ...
2    thought wonderful way spend time hot summer we...
3    basically family little boy jake think zombie ...
4    petter mattei love time money visually stunnin...
Name: clean_review, dtype: object


Considering positive as 1 and negative as 0

In [ ]:
vectorizer=CountVectorizer(max_features=5000)
x=vectorizer.fit_transform(df['clean_review']).toarray()
y=df['sentiment'].map({'positive':1,'negative':0})

Spliting the dataset into test and train

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

Training the model using logistic regression

In [ ]:
logistic_regression=LogisticRegression(max_iter=2000)
logistic_regression.fit(x_train,y_train)
y_pred_lr=logistic_regression.predict(x_test)
print("Logistic Regression - Acurracy Score: ",accuracy_score(y_test,y_pred_lr))
print(classification_report(y_test,y_pred_lr))

Logistic Regression - Acurracy Score:  0.8729
              precision    recall  f1-score   support

           0       0.88      0.87      0.87      5000
           1       0.87      0.88      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



Training the model using Naives Bayes

In [ ]:
nb=MultinomialNB()
nb.fit(x_train,y_train)
y_pred_nb=nb.predict(x_test)
print("Naive Bayes - Accuracy Score: ",accuracy_score(y_test,y_pred_nb))
print(classification_report(y_test,y_pred_nb))

Naive Bayes - Accuracy Score:  0.8495
              precision    recall  f1-score   support

           0       0.85      0.85      0.85      5000
           1       0.85      0.85      0.85      5000

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000



Training the model using Support Vector Machine

In [ ]:
svm=LinearSVC()
svm.fit(x_train,y_train)
y_pred_svm=svm.predict(x_test)
print("Support Vector Machine - Accuracy Score: ",accuracy_score(y_test,y_pred_svm))
print(classification_report(y_test,y_pred_svm))

Support Vector Machine - Accuracy Score:  0.8665
              precision    recall  f1-score   support

           0       0.87      0.86      0.87      5000
           1       0.86      0.87      0.87      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



Padding for deep learning and considering positive as 1 and negative as 0

In [ ]:
tokenizer=Tokenizer(num_words=1000,oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_review'])
x_seq=tokenizer.texts_to_sequences(df['clean_review'])
x_padded=pad_sequences(x_seq,maxlen=200,padding='post',truncating='post')
y=df['sentiment'].map({'positive':1,'negative':0})
x_train,x_test,t_train,y_test=train_test_split(x_padded,y,test_size=0.2,random_state=42)

Training The model using fully connected neural networks

In [ ]:
ann=Sequential([Embedding(input_dim=10000,output_dim=64,input_length=200),
                Flatten(),
                Dense(128,activation='relu'),
                Dropout(0.5),
                Dense(64,activation='relu'),
                Dense(1,activation='sigmoid')])
ann.compile(loss='binary_crossentropy',
            optimizer='adam',
            metrics=['accuracy'])
early_stop=EarlyStopping(monitor='val_loss',
                         patience=2,
                         restore_best_weights=True)
history_ann=ann.fit(x_train,y_train,epochs=5,batch_size=128,
                    validation_split=0.2,callbacks=[early_stop])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 18s 65ms/step - accuracy: 0.4986 - loss: 0.6963 - val_accuracy: 0.4978 - val_loss: 0.6934
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 58ms/step - accuracy: 0.5593 - loss: 0.6837 - val_accuracy: 0.4931 - val_loss: 0.7054
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 15s 60ms/step - accuracy: 0.7108 - loss: 0.5491 - val_accuracy: 0.4991 - val_loss: 0.8215


Training the model using Long short term memory

In [ ]:
lstm=Sequential([Embedding(input_dim=10000,output_dim=64,input_length=200),
                LSTM(128,dropout=0.2,recurrent_dropout=0.2),
                Dense(64,activation='relu'),
                Dropout(0.5),
                Dense(1,activation='sigmoid')])
lstm.compile(loss='binary_crossentropy',
            optimizer='adam',
            metrics=['accuracy'])
early_stop=EarlyStopping(monitor='val_loss',
                         patience=2,
                         restore_best_weights=True)
history_lstm=lstm.fit(x_train,y_train,epochs=10,batch_size=128,
                    validation_split=0.2,callbacks=[early_stop])

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 228s 900ms/step - accuracy: 0.5029 - loss: 0.6935 - val_accuracy: 0.4989 - val_loss: 0.6932
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 225s 900ms/step - accuracy: 0.4990 - loss: 0.6934 - val_accuracy: 0.5010 - val_loss: 0.6934
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 242s 969ms/step - accuracy: 0.5058 - loss: 0.6931 - val_accuracy: 0.4947 - val_loss: 0.6941


Training the modelusing convulutional neural network for text

In [ ]:
cnn=Sequential([Embedding(input_dim=10000,output_dim=64,input_length=200),
                Conv1D(128,5,activation='relu'),
                GlobalMaxPooling1D(),
                Dense(64,activation='relu'),
                Dropout(0.5),
                Dense(1,activation='sigmoid')])
cnn.compile(loss='binary_crossentropy',
            optimizer='adam',
            metrics=['accuracy'])
early_stop=EarlyStopping(monitor='val_loss',
                         patience=2,
                         restore_best_weights=True)
history_cnn=cnn.fit(x_train,y_train,epochs=10,batch_size=128,
                    validation_split=0.2,callbacks=[early_stop])

Epoch 1/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 40s 154ms/step - accuracy: 0.4993 - loss: 0.6939 - val_accuracy: 0.4994 - val_loss: 0.6932
Epoch 2/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 38s 153ms/step - accuracy: 0.5203 - loss: 0.6925 - val_accuracy: 0.4983 - val_loss: 0.6932
Epoch 3/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 42s 156ms/step - accuracy: 0.5473 - loss: 0.6892 - val_accuracy: 0.5016 - val_loss: 0.6939
Epoch 4/10
250/250 ━━━━━━━━━━━━━━━━━━━━ 39s 155ms/step - accuracy: 0.5961 - loss: 0.6730 - val_accuracy: 0.5038 - val_loss: 0.7034
